In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import pandas as pd
import numpy as np
import json, re
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score

In [5]:
class InterviewDataset(Dataset):
    def __init__(self, master_csv, jd_csv):
        df = pd.read_csv(master_csv)
        jd = pd.read_csv(jd_csv)
        df = df.merge(jd[['jd_id','jd_text_embedding']], on='jd_id', how='left')

        def parse_emb(s):
            try:
                return np.array(json.loads(s))
            except:
                parts = re.split(r'[,\s]+', s.strip().lstrip('[').rstrip(']'))
                return np.array([float(x) for x in parts if x])

        def parse_skills(s):
            return np.array(json.loads(s.replace("'", '"')))

        df['struct'] = df.apply(lambda r: np.concatenate([parse_skills(r['skills_vector']),
                                                           [r['segment_count']]]), axis=1)
        df['text']   = df['transcript_embedding'].apply(parse_emb)
        df['jd']     = df['jd_text_embedding'].   apply(parse_emb)

        df = df.dropna(subset=['struct','text','jd'])
        self.s = np.vstack(df['struct'].values).astype(np.float32)
        self.t = np.vstack(df['text'].values).  astype(np.float32)
        self.j = np.vstack(df['jd'].values).    astype(np.float32)

        self.y_reg = df['Overall_Score'].values.astype(np.float32)
        rec_cols = ['rec_Hire','rec_Consider','rec_Reject']
        self.y_cls = df[rec_cols].values.argmax(axis=1).astype(np.int64)

    def __len__(self):
        return len(self.y_reg)

    def __getitem__(self, idx):
        return (self.s[idx], self.j[idx], self.t[idx],
                self.y_reg[idx], self.y_cls[idx])

In [10]:
class GatedFusionModel(nn.Module):
    def __init__(self, dim_s, dim_j, dim_t, hidden, dropout):
        super().__init__()
        self.proj_s = nn.Sequential(nn.Linear(dim_s, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.proj_j = nn.Sequential(nn.Linear(dim_j, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.proj_t = nn.Sequential(nn.Linear(dim_t, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.gate_s = nn.Linear(dim_s, hidden)
        self.gate_j = nn.Linear(dim_j, hidden)
        self.gate_t = nn.Linear(dim_t, hidden)
        self.head_reg = nn.Linear(hidden, 1)
        self.head_cls = nn.Linear(hidden, 3)

    def forward(self, s, j, t):
        hs = self.proj_s(s)
        hj = self.proj_j(j)
        ht = self.proj_t(t)
        gs = torch.sigmoid(self.gate_s(s))
        gj = torch.sigmoid(self.gate_j(j))
        gt = torch.sigmoid(self.gate_t(t))
        fused = gs * hs + gj * hj + gt * ht
        return fused

    def predict(self, s, j, t):
        fused = self.forward(s, j, t)
        return self.head_reg(fused).squeeze(-1), self.head_cls(fused)

In [11]:
class CrossAttentionFusion(nn.Module):
    def __init__(self, dim_s, dim_j, dim_t, hidden, heads, dropout):
        super().__init__()
        # Projections
        self.proj_s = nn.Sequential(nn.Linear(dim_s, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.proj_j = nn.Sequential(nn.Linear(dim_j, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.proj_t = nn.Sequential(nn.Linear(dim_t, hidden), nn.ReLU(), nn.Dropout(dropout))
        # Self‑attention over the 3 modality tokens
        self.attn = nn.MultiheadAttention(embed_dim=hidden, num_heads=heads, dropout=dropout, batch_first=True)
        # Prediction heads
        self.head_reg = nn.Linear(hidden, 1)
        self.head_cls = nn.Linear(hidden, 3)

    def forward(self, s, j, t):
        hs = self.proj_s(s).unsqueeze(1)  # [B,1,H]
        hj = self.proj_j(j).unsqueeze(1)
        ht = self.proj_t(t).unsqueeze(1)
        seq = torch.cat([hs, hj, ht], dim=1)  # [B,3,H]
        attn_out, _ = self.attn(seq, seq, seq)
        fused = attn_out.mean(dim=1)  # [B,H]
        return fused

    def predict(self, s, j, t):
        fused = self.forward(s, j, t)
        return self.head_reg(fused).squeeze(-1), self.head_cls(fused)

In [13]:
# 1. Load dataset and split hold‑out
dataset = InterviewDataset('FINAL_master.csv', 'jds_clean.csv')
indices = np.arange(len(dataset))
_, test_idx = train_test_split(indices, test_size=0.1, random_state=42, stratify=dataset.y_cls)
test_ds = Subset(dataset, test_idx)
test_loader = DataLoader(test_ds, batch_size=32)

In [15]:
# 2. Load the concat (Full Fusion) scikit‑learn models
reg_concat = joblib.load("full_regressor.joblib")
clf_concat = joblib.load("full_classifier.joblib")


In [17]:
# 3. Load the PyTorch Gated and Cross‑Attention models
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dim_s, dim_j, dim_t = dataset.s.shape[1], dataset.j.shape[1], dataset.t.shape[1]

gated = GatedFusionModel(dim_s, dim_j, dim_t, hidden=256, dropout=0.1).to(device)
gated.load_state_dict(torch.load("gated_model.pth", map_location=device))
gated.eval()

attn = CrossAttentionFusion(dim_s, dim_j, dim_t, hidden=256, heads=2, dropout=0.45).to(device)
attn.load_state_dict(torch.load("cross_attention_model.pth", map_location=device))
attn.eval()


CrossAttentionFusion(
  (proj_s): Sequential(
    (0): Linear(in_features=35, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.45, inplace=False)
  )
  (proj_j): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.45, inplace=False)
  )
  (proj_t): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.45, inplace=False)
  )
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
  )
  (head_reg): Linear(in_features=256, out_features=1, bias=True)
  (head_cls): Linear(in_features=256, out_features=3, bias=True)
)

In [18]:
# 4. Ensemble predictions
y_true_reg, y_pred_reg, y_true_cls, y_pred_cls = [], [], [], []

for s, j, t, y_reg_true, y_cls_true in test_loader:
    # record ground truth
    y_true_reg.extend(y_reg_true.numpy())
    y_true_cls.extend(y_cls_true.numpy())

    # move to device
    s, j, t = s.to(device), j.to(device), t.to(device)

    # --- Concat baseline (scikit-learn) ---
    feats = np.hstack([s.cpu().numpy(), j.cpu().numpy(), t.cpu().numpy()])
    r_concat = reg_concat.predict(feats)             # (batch,)
    p_concat = clf_concat.predict_proba(feats)       # (batch, 3)

    # --- Gated fusion (PyTorch) ---
    pr_g, pc_g = gated.predict(s, j, t)
    r_g = pr_g.cpu().detach().numpy()
    p_g = F.softmax(pc_g, dim=1).cpu().detach().numpy()

    # --- Cross-attention (PyTorch) ---
    pr_a, pc_a = attn.predict(s, j, t)
    r_a = pr_a.cpu().detach().numpy()
    p_a = F.softmax(pc_a, dim=1).cpu().detach().numpy()

    # Regression: average of three
    r_ens = (r_concat + r_g + r_a) / 3
    y_pred_reg.extend(r_ens.tolist())

    # Classification: average softmax probs
    p_ens = (p_concat + p_g + p_a) / 3
    y_pred_cls.extend(np.argmax(p_ens, axis=1).tolist())

In [20]:
# 5. Compute final metrics
rmse = mean_squared_error(y_true_reg, y_pred_reg)
rmse = np.sqrt(rmse)
acc  = accuracy_score(y_true_cls, y_pred_cls)
print(f"► Ensemble Test RMSE: {rmse:.4f}")
print(f"► Ensemble Test Accuracy: {acc:.4f}")

► Ensemble Test RMSE: 0.0680
► Ensemble Test Accuracy: 0.8710
